In [1]:
import sys
from pathlib import Path
root = Path.cwd().parents[1]
sys.path.append(str(root))

In [2]:
from pprint import pprint
import os
import logging

debug = False
logger = logging.getLogger()
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

from src.preprocessing import (
    Reader, HTTPReader,
    Embedder, HTTPEmbedder,
    FixedCharChunker,
    Document, Page
)
from src.indexing import DataStore, ChunkStore, FlatVectorStore, Index
from src.utils import AutoEDAIndex
from src.llm import OpenAILLM

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Инициализация индекса

In [ ]:
# embedder = HTTPEmbedder()
# embedder.get_memory_usage()

{'device': 'mps:0',
 'cpu_rss_mb': 625.203125,
 'mps_current_allocated_mb': 3141.0263671875,
 'mps_driver_allocated_mb': 3710.640625}

In [ ]:
# reader = HTTPReader()
# res = reader.read('/Users/danilaandreev/Documents/HSE/degree/data/raw_data/4293720391.pdf')

In [ ]:
DATA_STORE_DIR = '../../data/index/documents'
CHUNK_STORE_DIR = '../../data/index/chunks'
VECTOR_STORE_DIR = '../../data/index/vectors'
SQLITE_DIR = '../../data/index/db'

# embedder = Embedder(
#     device='mps', 
#     model_name='ai-forever/FRIDA',
#     type_handlers={
#         'query': lambda text: f'search_query: {text}',
#         'document': lambda text: f'search_document: {text}',
#     }
# )
# reader=Reader(logger)

embedder = HTTPEmbedder()
reader=HTTPReader(logger)

dimensions = embedder.get_embeddings(['test']).shape[-1]

index = Index(
    datastore=DataStore(DATA_STORE_DIR, logger),
    vectorstore=FlatVectorStore(VECTOR_STORE_DIR, dimensions, logger),
    chunkstore=ChunkStore(CHUNK_STORE_DIR, logger),
    chunker=FixedCharChunker(chunk_size=500, overlap=50, logger=logger),
    embedder=embedder,
    reader=reader,
    sqlite_path=SQLITE_DIR,
    logger=logger
)

2026-04-03 09:06:38,926 | INFO | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: ai-forever/FRIDA
2026-04-03 09:07:59,255 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/ai-forever/FRIDA/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-03 09:07:59,255 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-03 09:07:59,287 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ai-forever/FRIDA/aed004da8d09c33f6a51240fd9f5bcc625225b67/modules.json "HTTP/1.1 200 OK"
2026-04-03 09:07:59,449 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/ai-forever/FRIDA/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-03 09:07:59,479 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ai-forever/FRIDA/ae

In [4]:
# index.clear()
index.info()

2026-04-03 09:08:42,138 | INFO | root | DataStore data removed from ../../data/index/documents
2026-04-03 09:08:49,423 | INFO | root | ChunkStore data removed from ../../data/index/chunks
2026-04-03 09:08:49,429 | INFO | root | VectorStore data removed from ../../data/index/vectors
2026-04-03 09:08:49,525 | INFO | root | Index data removed
2026-04-03 09:08:49,526 | INFO | root | Index stats: {'datastore': {'documents': 0}, 'chunkstore': {'chunks': 0}, 'vectorstore': {'vectors': 0, 'dimensions': 1536}, 'index': {'points': 0, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 0},
 'chunkstore': {'chunks': 0},
 'vectorstore': {'vectors': 0, 'dimensions': 1536},
 'index': {'points': 0, 'points_without_chunk': 0, 'chunks_without_point': 0}}

## Загрузка документов

In [5]:
SOURCE_DATA_DIR = '../../data/raw_data'

sources = [os.path.join(SOURCE_DATA_DIR, name) for name in os.listdir(SOURCE_DATA_DIR)]
print(f'Количество источников: {len(sources)}')

doc_ids = index.add_sources(sources)
index.save_vectorstore()

2026-04-03 09:09:26,410 | INFO | root | add_sources: 175 new sources, 0 skipped


Количество источников: 175


Indexing 4293732872.pdf | RSS 905 MB | MPS 3141/3698 MB:  15%|█▌        | 27/175 [06:20<45:57, 18.63s/it]  2026-04-03 09:15:47,203 | INFO | root | File 2a46d2d4a69bc6af3019e8d1bd488bbfc4f0529b09e81a31054ac03b9aa6a50e.json already exists
2026-04-03 09:15:47,216 | INFO | root | File d9a491e85dcf56f4fc0f48f0866ed4a41070218faeaefbf555dcef0e719689da.json already exists
2026-04-03 09:15:47,218 | INFO | root | File d9a491e85dcf56f4fc0f48f0866ed4a41070218faeaefbf555dcef0e719689da.json already exists
2026-04-03 09:15:47,219 | INFO | root | File 9347b4d9c31ba22ca642998a4001261684837acb80a01c84705335f5856ed15c.json already exists
2026-04-03 09:15:47,219 | INFO | root | File d9a491e85dcf56f4fc0f48f0866ed4a41070218faeaefbf555dcef0e719689da.json already exists
2026-04-03 09:15:47,220 | INFO | root | File d9a491e85dcf56f4fc0f48f0866ed4a41070218faeaefbf555dcef0e719689da.json already exists
2026-04-03 09:15:47,221 | INFO | root | File d9a491e85dcf56f4fc0f48f0866ed4a41070218faeaefbf555dcef0e719689da.jso

In [12]:
index.info()

2026-04-03 19:06:07,911 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 58762}, 'vectorstore': {'vectors': 58980, 'dimensions': 1536}, 'index': {'points': 58762, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 58762},
 'vectorstore': {'vectors': 58980, 'dimensions': 1536},
 'index': {'points': 58762,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

In [7]:
test_query = 'Нагрузка перекрытий'
res = index.search(test_query)

for r in res:
    print(r.chunk['text'])
    print()

редств Правила настоящего раздела распространяются на нагрузки от людей, животных, оборудования, изделий, материалов, временных перегородок, транспортных средств, действующие на перекрытия, покрытия, лестницы зданий и сооружений и полы на грунтах. Варианты загружения перекрытий этими нагрузками следует принимать в соответствии с предусмотренными условиями возведения и эксплуатации зданий, в наиболее неблагоприятном расчетном положении. Если на стадии проектирования данные об этих условиях недост

таны на восприятие постоянных нагрузок: - от собственного веса несущих и ограждающих конструкций; - временных равномерно распределенных и сосредоточенных нагрузок на перекрытия; - снеговых и ветровых нагрузок для данного района строительства. Нормативные значения перечисленных нагрузок, учитываемые неблагоприятные сочетания на­ грузок или соответствующих им усилий, предельные значения прогибов и перемещений конструкций, значения коэффициентов надежности по нагрузкам должны быть приняты в соотв

# Baseline

In [8]:
eda = AutoEDAIndex(index)
df = eda.run()
# df

DOCUMENTS
- count: 175
- text_length_mean: 140687.2857142857
- text_length_median: 108903.0
- text_length_min: 1874
- text_length_max: 693714

CHUNKS
- count: 58762
- text_length_mean: 457.77097443926345
- text_length_median: 500.0
- text_length_min: 51
- text_length_max: 500
- text_length_std: 104.0179440315079
- words_mean: 63.612691875701984
- sentences_mean: 5.673564548517749
- paragraphs_mean: 1.0

EMBEDDINGS
- matrix_rows: 58980
- matrix_cols: 1536
- dtype: float64
- dimension_std_mean: 0.021747533816172542
- dimension_std_min: 0.0
- dimension_std_max: 0.028831579473439874
- pairwise_similarity_pairs: 1739290710
- pairwise_similarity_min: -0.20393946068779106
- pairwise_similarity_max: 1.0000001802826095
- pairwise_similarity_mean: 0.25337309167476496
- pairwise_similarity_std: 0.08861156391317566



In [9]:
from pydantic import BaseModel, Field
from typing import Optional, List
    
llm = OpenAILLM(client, model_name='gpt-5.4')

# class Answer(BaseModel):
#     answer: str
# res = llm.parse('Скажи привет', Answer)
# print(res)

In [10]:
query = 'Какие бывают нагрузки на перекрытия в многоэтажных зданиях?'
extracted = index.search(query, top_k=10)
# context = '\n\n'.join([f'{i+1}. Doc {e.chunk['doc_id']}: page {e.chunk['page_number']}\n{e.chunk['text']}' for i, e in enumerate(extracted)])
context = '\n\n'.join([f'{i+1}. {e.chunk['text']}' for i, e in enumerate(extracted)])
prompt = f'Ты отвечаешь на вопросы пользователя по строительной документации. Отвечай только на основе доступного контекста.\n\nЗапрос: {query}\n\nКонтекст: {context}'
class Answer(BaseModel):
    reasoning: List[str] = Field(..., description='3-5 предложений по обдумыванию контекста')
    answer: str = Field(..., description='Финальный ответ')
answer = llm.parse(prompt, Answer)
pprint(answer.answer)

2026-04-03 10:14:31,885 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


('На перекрытия в многоэтажных зданиях по приведенному контексту действуют:\n'
 '\n'
 '- постоянные нагрузки — от собственного веса несущих и ограждающих '
 'конструкций;\n'
 '- временные равномерно распределенные нагрузки на перекрытия;\n'
 '- временные сосредоточенные нагрузки на перекрытия;\n'
 '- нагрузки от людей, оборудования, изделий, материалов, временных '
 'перегородок, транспортных средств;\n'
 '- в составе общих расчетов здания также учитываются снеговые и ветровые '
 'нагрузки.\n'
 '\n'
 'Для перекрытий и лестниц в контексте отдельно указана сосредоточенная '
 'нагрузка 1,5 кН, если в задании на проектирование не предусмотрены более '
 'высокие значения.')


In [11]:
print(context)

1. редств Правила настоящего раздела распространяются на нагрузки от людей, животных, оборудования, изделий, материалов, временных перегородок, транспортных средств, действующие на перекрытия, покрытия, лестницы зданий и сооружений и полы на грунтах. Варианты загружения перекрытий этими нагрузками следует принимать в соответствии с предусмотренными условиями возведения и эксплуатации зданий, в наиболее неблагоприятном расчетном положении. Если на стадии проектирования данные об этих условиях недост

2. таны на восприятие постоянных нагрузок: - от собственного веса несущих и ограждающих конструкций; - временных равномерно распределенных и сосредоточенных нагрузок на перекрытия; - снеговых и ветровых нагрузок для данного района строительства. Нормативные значения перечисленных нагрузок, учитываемые неблагоприятные сочетания на­ грузок или соответствующих им усилий, предельные значения прогибов и перемещений конструкций, значения коэффициентов надежности по нагрузкам должны быть приняты в

In [ ]:
# TODO
# проверить бенчмарк
# сделать оценку через бенчмарк
# зафиксировать результаты
# 

После создания бенчмарка требуется сделать таблицу, которая будет показывать все основные метрики качества при различных комбинациях создания индекса. 
- Сначала сделаем бейзлайн
- Потом попробуем перебрать разные варианты чанкирования - выберем лучший для по метрикам
- Потом переберем разные варианты моделей - выберем лучшие
- Потом будет пробовать изменить архитектуру - выбираем лучшую
- Потом допиливаем через дообучение моделей
Всё должно собираться в dataframe для сохранения результатов